<a href="https://www.kaggle.com/code/jayhawk1900/f1-3way-blend?scriptVersionId=321471026" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# F1 Pit Stop Prediction — 3-Way Blend (RealMLP + TabM + XGBoost)

**Competition:** [Playground Series S6E5](https://www.kaggle.com/competitions/playground-series-s6e5) | **Metric:** ROC-AUC

---

## Overview

A cross-family ensemble combining three models with uncorrelated errors:

1. **RealMLP** (neural, PBLD periodic embeddings, 16-member internal ensemble) — trained here, 5 epochs × 3 seeds, OOF ≈ 0.954
2. **TabM** (neural, BatchEnsemble parameter-efficient ensemble) — predictions loaded from a prior run, OOF ≈ 0.951
3. **XGBoost** (gradient-boosted trees, Optuna-tuned, ORIG-concatenated) — predictions loaded, OOF ≈ 0.951

All three use the original F1 strategy dataset concatenated as training data. Predictions are **rank-normalized** before blending (lossless for ROC-AUC, equalizes scale across differently-calibrated models), then combined via **coordinate-ascent** weight optimization. The notebook reports inter-model correlations to show how much fresh signal each contributes.

The project's recurring lesson: diversity across model families beats depth within one. The two neural architectures use different ensembling mechanisms (independent networks vs. shared backbone with rank-1 adapters), so even both being neural, they add complementary signal.

In [1]:
# ============================================================
# 3-WAY BLEND: train RealMLP, load TabM + XGB, blend all three
# ============================================================
import os, math, random, time, warnings
import numpy as np, pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import KBinsDiscretizer, TargetEncoder
import torch
import torch.nn as nn
warnings.filterwarnings('ignore')

def seed_everything(seed):
    np.random.seed(seed); random.seed(seed); torch.manual_seed(seed)
seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| device:', device)

# ---------- Paths to pre-computed TabM and XGB predictions ----------
TABM_DIR = '/kaggle/input/notebooks/jayhawk1900/f1-tabm'
XGB_DIR  = '/kaggle/input/notebooks/jayhawk1900/f1-final-blend'
# Verify they exist before the long training run
for f in [f'{TABM_DIR}/oof_tabm.npy', f'{TABM_DIR}/pred_tabm.npy',
          f'{XGB_DIR}/oof_xgb_orig.npy', f'{XGB_DIR}/pred_xgb_orig.npy']:
    assert os.path.exists(f), f'MISSING: {f}'
print('TabM and XGB prediction files confirmed present.')

# ---------- Load data ----------
DATA_DIR = '/kaggle/input/competitions/playground-series-s6e5'
ORIG_DIR = '/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction'
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')
orig = pd.read_csv(f'{ORIG_DIR}/f1_strategy_dataset_v4.csv')
print('Loaded:', train.shape, test.shape, orig.shape)

ID, TARGET = 'id', 'PitNextLap'
orig = orig.drop(['Normalized_TyreLife'], axis=1)
y_orig = orig[TARGET]; orig = orig.drop([TARGET], axis=1)
X = train.drop([ID, TARGET], axis=1); train_id = train[ID]
y = train[TARGET]
X_test = test.drop([ID], axis=1); test_id = test[ID]
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object']).columns.tolist()

# ---------- Feature engineering ----------
category_map = {}
important_combos = [('Race', 'Compound'), ('Race', 'Year')]

def feature_engineering(df, fit=False):
    df = df.copy()
    df['_LapNumber_/_RaceProgress'] = (df['LapNumber'] / (df['RaceProgress'] + 1e-6)).astype('float32')
    df['_TyreLife_/_LapNumber'] = (df['TyreLife'] / df['LapNumber'].clip(lower=1)).astype('float32')
    df['_LapTime (s)_*_Cumulative_Degradation'] = (df['LapTime (s)'] * df['Cumulative_Degradation']).astype('float32')
    df['_LapTime (s)_*_Cumulative_Degradation_abs'] = (df['LapTime (s)'] * df['Cumulative_Degradation'].abs()).astype('float32')
    df['_LapTime (s)_/_Cumulative_Degradation_abs'] = (df['LapTime (s)'] / (df['Cumulative_Degradation'].abs() + 1e-6)).astype('float32')
    for col in cat_cols:
        if fit:
            codes, uniques = df[col].factorize(); category_map[col] = uniques
        else:
            uniques = category_map[col]; cm = {c: i for i, c in enumerate(uniques)}
            codes = df[col].map(cm).fillna(-1).astype('int32')
        df[col] = pd.Series(codes, index=df.index).astype('category')
    for col in num_cols + ['_LapNumber_/_RaceProgress', '_TyreLife_/_LapNumber']:
        cn = f"{col}_cat_" if col in num_cols else f"{col[1:]}_cat_"
        if fit:
            codes, uniques = np.floor(df[col]).factorize(); category_map[col] = uniques
        else:
            uniques = category_map[col]; cm = {c: i for i, c in enumerate(uniques)}
            codes = np.floor(df[col]).map(cm).fillna(-1).astype('int32')
        df[cn] = pd.Series(codes, index=df.index).astype('category')
    for col in cat_cols + ['Year_cat_', 'PitStop_cat_']:
        nm = f"_{col}_count" if col in cat_cols else f"_{col[:-1]}_count"
        if fit:
            cmap = df[col].value_counts(); category_map[nm] = cmap
        else:
            cmap = category_map[nm]
        df[nm] = df[col].astype(object).map(cmap).fillna(0).astype('int32')
    for col, bins_list in {'RaceProgress': [200], 'LapTime (s)': [7]}.items():
        for nb in bins_list:
            bn = f"{col}_{nb}_quantile_bin_"
            if fit:
                kb = KBinsDiscretizer(n_bins=nb, encode='ordinal', strategy='quantile', subsample=None)
                b = kb.fit_transform(df[[col]]).ravel().astype('int32'); category_map[bn] = kb
            else:
                kb = category_map[bn]; b = kb.transform(df[[col]]).ravel().astype('int32')
            df[bn] = pd.Series(b, index=df.index).astype('category')
    combo_names = []
    for cols in important_combos:
        cn = '_'.join(cols) + '_'; combo_names.append(cn)
        s = df[cols[0]].astype(str)
        for c in cols[1:]: s = s + '_' + df[c].astype(str)
        if fit:
            codes, uniques = pd.factorize(s, sort=False); category_map[cn] = uniques
        else:
            uniques = category_map[cn]; cm = {c: i for i, c in enumerate(uniques)}
            codes = s.map(cm).fillna(-1).astype('int32')
        df[cn] = pd.Series(codes, index=df.index).astype('category')
    new_cat = [c for c in df.columns if c.endswith('_')]
    new_num = [c for c in df.columns if c.startswith('_')]
    return df, new_cat, new_num, combo_names

X, new_cat_cols, new_num_cols, combo_names = feature_engineering(X, fit=True)
X_test, _, _, _ = feature_engineering(X_test, fit=False)
orig, _, _, _ = feature_engineering(orig, fit=False)
cat_cols_all = cat_cols + new_cat_cols
print('After FE:', X.shape, '| cat:', len(cat_cols_all))

# ---------- RealMLP components ----------
class NumericalPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, tfms):
        self._tfms = [t for t in tfms if t in ("median_center","robust_scale","smooth_clip")]
    def fit(self, X, y=None):
        self._median = np.median(X, axis=0)
        q = np.quantile(X,0.75,axis=0)-np.quantile(X,0.25,axis=0)
        z = q==0.0; q[z]=0.5*(X.max(axis=0)[z]-X.min(axis=0)[z])
        self._iqr=1.0/(q+1e-30); self._iqr[q==0.0]=0.0
        return self
    def transform(self, X, y=None):
        X = X.copy().astype(np.float32)
        for t in self._tfms:
            if t=="median_center": X-=self._median[None,:]
            elif t=="robust_scale": X*=self._iqr[None,:]
            elif t=="smooth_clip": X=X/np.sqrt(1+(X/3)**2)
        return X

class CategoricalFeatureLayer(nn.Module):
    def __init__(self, n_ens, cat_dims, embed_dim=8, onehot_thresh=8):
        super().__init__()
        self.n_ens=n_ens; self.cat_dims=cat_dims; self.onehot_features=[]
        self.embed_layers=nn.ModuleList(); self._embed_feature_indices=[]
        for i,dim in enumerate(cat_dims):
            if dim<=onehot_thresh: self.onehot_features.append(i)
            else:
                self.embed_layers.append(nn.ModuleList([nn.Embedding(dim,embed_dim) for _ in range(n_ens)]))
                self._embed_feature_indices.append(i)
    def forward(self,x):
        b,k,_=x.shape; feats=[]
        if self.onehot_features:
            ox=x[:,:,self.onehot_features]; od=[self.cat_dims[i] for i in self.onehot_features]
            enc=torch.zeros(b,k,sum(od),device=x.device); st=0
            for idx,dim in enumerate(od):
                enc.scatter_(2, ox[:,:,idx:idx+1].long()+st, 1.0); st+=dim
            feats.append(enc)
        for el,fi in zip(self.embed_layers,self._embed_feature_indices):
            fe=[el[m](x[:,m,fi:fi+1].long()) for m in range(self.n_ens)]
            feats.append(torch.cat(fe,dim=1))
        return torch.cat(feats,dim=2)

class ScalingLayer(nn.Module):
    def __init__(self,n_ens,n_features):
        super().__init__(); self.scale=nn.Parameter(torch.ones(n_ens,n_features))
    def forward(self,x): return x*self.scale[None,:,:]

class NTPLinear(nn.Module):
    def __init__(self,n_ens,in_f,out_f,bias=True):
        super().__init__(); self.in_features=in_f
        self.weight=nn.Parameter(torch.randn(n_ens,in_f,out_f))
        self.bias=nn.Parameter(torch.randn(n_ens,out_f)) if bias else None
    def forward(self,x):
        x=torch.einsum("bki,kio->bko",x,self.weight)/math.sqrt(self.in_features)
        if self.bias is not None: x=x+self.bias
        return x

class PBLDEmbedding(nn.Module):
    def __init__(self,n_ens,n_features,hidden_dim=16,out_dim=4,freq_scale=0.1,activation=nn.GELU):
        super().__init__(); self.n_ens=n_ens; self.n_features=n_features; self.out_dim=out_dim
        self.w1=nn.Parameter(torch.randn(n_ens,n_features,hidden_dim)*freq_scale)
        self.b1=nn.Parameter(torch.randn(n_ens,n_features,hidden_dim))
        self.w2=nn.Parameter(torch.randn(n_ens,n_features,hidden_dim,out_dim-1)/math.sqrt(hidden_dim))
        self.b2=nn.Parameter(torch.randn(n_ens,n_features,out_dim-1))
        self.act=activation(); nn.init.uniform_(self.b1,-math.pi,math.pi)
    def forward(self,x):
        per=torch.cos(2*math.pi*(x.unsqueeze(-1)*self.w1.unsqueeze(0)+self.b1.unsqueeze(0)))
        tr=self.act(torch.einsum("bkfh,kfhd->bkfd",per,self.w2)+self.b2.unsqueeze(0))
        return torch.cat([x.unsqueeze(-1),tr],dim=-1).flatten(start_dim=2)

class RealMLP(nn.Module):
    def __init__(self,output_dim,cat_dims,n_numerical,cfg):
        super().__init__()
        n_ens=cfg["n_ens"]; embed_dim=cfg["embed_dim"]; self.n_ens=n_ens
        self.cate=CategoricalFeatureLayer(n_ens,cat_dims,embed_dim,cfg["onehot_thresh"])
        self.num_embed=PBLDEmbedding(n_ens,n_numerical,cfg["pbld_hidden_dim"],
                                     cfg["pbld_out_dim"],cfg["pbld_freq_scale"],cfg["pbld_activation"])
        num_emb=n_numerical*cfg["pbld_out_dim"]
        cat_emb=sum(c if c<=cfg["onehot_thresh"] else embed_dim for c in cat_dims)
        total=num_emb+cat_emb; act=cfg["activation"]
        layers=[]
        if cfg["add_front_scale"]: layers.append(ScalingLayer(n_ens,total))
        self._dropout_modules=[]; in_dim=total
        for i,h in enumerate(cfg["hidden_dims"]):
            lin=NTPLinear(n_ens,in_dim,h)
            if i==0: self.first_linear=lin
            drop=nn.Dropout(cfg["dropout"]); self._dropout_modules.append(drop)
            layers+=[lin,act(),drop]; in_dim=h
        self.hidden=nn.Sequential(*layers)
        self.output_layer=NTPLinear(n_ens,in_dim,output_dim)
    def forward(self,x_num,x_cat):
        x_num=x_num.unsqueeze(1).expand(-1,self.n_ens,-1)
        x_cat=x_cat.unsqueeze(1).expand(-1,self.n_ens,-1)
        x_num=self.num_embed(x_num); x_cat=self.cate(x_cat)
        x=self.hidden(torch.cat([x_num,x_cat],dim=2))
        return torch.sigmoid(self.output_layer(x))

def apply_schedule(v,p,sched,flat_ratio=0.3):
    if sched=="constant": return v
    if sched=="cos": return v*(math.cos(math.pi*p)+1)/2
    if sched=="flat_cos":
        if p<flat_ratio: return v
        t=(p-flat_ratio)/(1-flat_ratio); return v*(math.cos(math.pi*t)+1)/2
    if sched=="expm4t": return v*math.exp(-4*p)
    raise ValueError(sched)

def get_parameter_groups(model,p):
    fid=id(model.first_linear.weight)
    sp,pp,fp,op,bp=[],[],[],[],[]
    for name,par in model.named_parameters():
        if "num_embed" in name: pp.append(par)
        elif "scale" in name: sp.append(par)
        elif id(par)==fid: fp.append(par)
        elif "bias" in name: bp.append(par)
        else: op.append(par)
    LR=p["lr"]; WD=p["weight_decay"]
    return [
        {"params":sp,"lr":LR*p["lr_scale_mult"],"weight_decay":WD*p["wd_scale_mult"]},
        {"params":pp,"lr":LR*p["pbld_lr_factor"],"weight_decay":WD},
        {"params":fp,"lr":LR*p["first_layer_lr_factor"],"weight_decay":WD},
        {"params":op,"lr":LR,"weight_decay":WD},
        {"params":bp,"lr":LR*p["lr_bias_mult"],"weight_decay":WD*p["wd_bias_mult"]},
    ]

def binary_bce_loss(yt,yp,ls=0.0,pos_weight=None,eps=1e-7):
    if ls>0.0: yt=yt*(1.0-ls)+0.5*ls
    yp=yp.clamp(eps,1.0-eps)
    if pos_weight is None: loss=-yt*torch.log(yp)-(1.0-yt)*torch.log(1.0-yp)
    else: loss=-pos_weight*yt*torch.log(yp)-(1.0-yt)*torch.log(1.0-yp)
    return loss.mean()

class RealMLP_TD_Classifier(BaseEstimator):
    def __init__(self,**kw): self.params={**CONFIG,**kw}
    def fit(self,X_train,y_train,X_val,y_val,cat_col_names=None,ckpt_path="ckpt.pth",X_test=None):
        p=self.params; dev=device; vb=p["verbosity"]; cat_col_names=cat_col_names or []
        ncn=[c for c in X_train.columns if c not in cat_col_names]
        Xtn=X_train[ncn].values.astype(np.float32); Xvn=X_val[ncn].values.astype(np.float32)
        Xtc=X_train[cat_col_names].values.astype(np.int64); Xvc=X_val[cat_col_names].values.astype(np.int64)
        y_tr=np.asarray(y_train); y_v=np.asarray(y_val)
        self.prep=NumericalPreprocessor(p["tfms"]).fit(Xtn)
        Xtn=self.prep.transform(Xtn); Xvn=self.prep.transform(Xvn)
        self.cat_=cat_col_names; self.num_=ncn
        allc=[Xtc,Xvc]
        if X_test is not None: allc.append(X_test[cat_col_names].values.astype(np.int64))
        cat_dims=(np.concatenate(allc,axis=0).max(axis=0)+1).tolist()
        self.cat_dims_=cat_dims; cmax=np.array(cat_dims)-1
        Xtc=np.clip(Xtc,0,cmax); Xvc=np.clip(Xvc,0,cmax)
        w=compute_class_weight("balanced",classes=np.unique(y_tr),y=y_tr)
        pw=torch.tensor(w[1],dtype=torch.float32,device=dev)
        self.model=RealMLP(1,cat_dims,Xtn.shape[1],p).to(dev)
        pg=get_parameter_groups(self.model,p)
        for g in pg: g["lr_base"]=g["lr"]
        opt=torch.optim.AdamW(pg,betas=(p["mom"],p["sq_mom"]))
        Xtn_t=torch.as_tensor(Xtn,device=dev); Xtc_t=torch.as_tensor(Xtc,dtype=torch.long,device=dev)
        ytt=torch.as_tensor(y_tr,dtype=torch.float32,device=dev)
        Xvn_t=torch.as_tensor(Xvn,device=dev); Xvc_t=torch.as_tensor(Xvc,dtype=torch.long,device=dev)
        n_ens=p["n_ens"]; tbs=p["train_bs"]; ebs=p["eval_bs"]; ep=p["epochs"]
        lrs=p["lr_sched"]; fr=p["flat_ratio"]; total=ep*len(y_tr); order=np.arange(len(y_tr))
        best=-np.inf; best_vp=None
        for epoch in range(ep):
            self.model.train()
            for st in range(0,len(y_tr),tbs):
                prog=(epoch*len(y_tr)+st)/total; idx=order[st:st+tbs]
                for g in opt.param_groups: g["lr"]=apply_schedule(g["lr_base"],prog,lrs,fr)
                opt.zero_grad()
                yp=self.model(Xtn_t[idx],Xtc_t[idx])
                ls=apply_schedule(p["ls_eps"],prog,p["ls_eps_sched"],fr)
                dv=apply_schedule(p["dropout"],prog,p["p_drop_sched"],fr)
                for dm in self.model._dropout_modules: dm.p=dv
                loss=binary_bce_loss(ytt[idx].repeat_interleave(n_ens),yp.reshape(-1),ls=ls,pos_weight=pw)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(),p["grad_clip"]); opt.step()
            np.random.shuffle(order)
            self.model.eval()
            with torch.no_grad():
                vp=np.concatenate([self.model(Xvn_t[s:s+ebs],Xvc_t[s:s+ebs]).mean(dim=1).squeeze(-1).cpu().numpy()
                                   for s in range(0,len(y_v),ebs)])
            sc=roc_auc_score(y_v,vp)
            if sc>best: best=sc; best_vp=vp.copy(); torch.save(self.model.state_dict(),ckpt_path)
            if vb>=2: print(f"    epoch {epoch+1}/{ep} score={sc:.5f} best={best:.5f}")
        self.model.load_state_dict(torch.load(ckpt_path))
        self.best_val_probs_=best_vp
        return self
    def predict_proba(self,X):
        ebs=self.params["eval_bs"]
        Xn=self.prep.transform(X[self.num_].values.astype(np.float32))
        Xc=np.clip(X[self.cat_].values.astype(np.int64),0,np.array(self.cat_dims_)-1)
        Xn=torch.as_tensor(Xn,device=device); Xc=torch.as_tensor(Xc,dtype=torch.long,device=device)
        self.model.eval()
        with torch.no_grad():
            pp=np.concatenate([self.model(Xn[s:s+ebs],Xc[s:s+ebs]).mean(dim=1).squeeze(-1).cpu().numpy()
                               for s in range(0,len(Xn),ebs)])
        return np.stack([1-pp,pp],axis=1)

CONFIG = {
    "n_ens":16,"embed_dim":6,"onehot_thresh":4,"hidden_dims":[512,64,128],
    "dropout":0.05,"p_drop_sched":"expm4t","activation":nn.SiLU,"add_front_scale":True,
    "pbld_hidden_dim":20,"pbld_out_dim":5,"pbld_freq_scale":5.0,
    "pbld_activation":nn.PReLU,"pbld_lr_factor":0.093,
    "lr":0.008,"mom":0.9,"sq_mom":0.98,"lr_sched":"flat_cos","flat_ratio":0.3,
    "first_layer_lr_factor":1.0,"lr_scale_mult":10.0,"lr_bias_mult":0.1,
    "weight_decay":0.005,"wd_scale_mult":0.1,"wd_bias_mult":0.5,"grad_clip":1.0,
    "ls_eps":0.04,"ls_eps_sched":"cos","tfms":["median_center","robust_scale","smooth_clip"],
    "epochs":5,"train_bs":256,"eval_bs":10240,"verbosity":2,"device":"cuda",
}

# ---------- Train RealMLP (5 epochs, 3 seeds) ----------
FOLDS=5; SEEDS=[42,123,7]
oof_rmlp=np.zeros(len(X)); pred_rmlp=np.zeros(len(X_test))
t_all=time.time()
for si,S in enumerate(SEEDS):
    print(f'\n=== RealMLP SEED {S} ({si+1}/{len(SEEDS)}) ===')
    seed_everything(S)
    skf=StratifiedKFold(n_splits=FOLDS,shuffle=True,random_state=S)
    oof_s=np.zeros(len(X)); pred_s=np.zeros(len(X_test))
    for fold,((tr,va),(otr,_)) in enumerate(zip(skf.split(X,y),skf.split(orig,y_orig)),1):
        X_tr=pd.concat([X.iloc[tr],orig.iloc[otr]]).reset_index(drop=True)
        y_tr=pd.concat([y.iloc[tr],y_orig.iloc[otr]]).reset_index(drop=True)
        X_val=X.iloc[va].copy(); y_val=y.iloc[va]; X_tst=X_test.copy()
        te=TargetEncoder(cv=FOLDS,smooth='auto',shuffle=True,random_state=S)
        X_tr_te=te.fit_transform(X_tr[combo_names],y_tr)
        X_val_te=te.transform(X_val[combo_names]); X_tst_te=te.transform(X_tst[combo_names])
        ten=[f"_{c}TE" for c in combo_names]
        X_tr[ten]=X_tr_te; X_val[ten]=X_val_te; X_tst[ten]=X_tst_te
        m=RealMLP_TD_Classifier(**CONFIG)
        m.fit(X_tr,y_tr,X_val,y_val,cat_col_names=cat_cols_all,ckpt_path=f'r_s{S}_f{fold}.pth')
        oof_s[va]=m.best_val_probs_
        pred_s+=m.predict_proba(X_tst)[:,1]/FOLDS
        print(f'  Seed {S} Fold {fold}: {roc_auc_score(y_val,m.best_val_probs_):.5f}')
        torch.cuda.empty_cache()
    print(f'>>> Seed {S} OOF: {roc_auc_score(y,oof_s):.5f}')
    oof_rmlp+=oof_s/len(SEEDS); pred_rmlp+=pred_s/len(SEEDS)
rmlp_auc=roc_auc_score(y,oof_rmlp)
print(f'\nRealMLP multi-seed OOF: {rmlp_auc:.5f}  ({(time.time()-t_all)/60:.1f} min)')
np.save('/kaggle/working/oof_realmlp_ms5.npy',oof_rmlp)
np.save('/kaggle/working/pred_realmlp_ms5.npy',pred_rmlp)

# ---------- Load TabM and XGB ----------
oof_tabm=np.load(f'{TABM_DIR}/oof_tabm.npy'); pred_tabm=np.load(f'{TABM_DIR}/pred_tabm.npy')
oof_xgb=np.load(f'{XGB_DIR}/oof_xgb_orig.npy'); pred_xgb=np.load(f'{XGB_DIR}/pred_xgb_orig.npy')

# ---------- Correlations + 3-way blend ----------
def rn(x): return rankdata(x)/len(x)
r_rmlp,r_tabm,r_xgb=rn(oof_rmlp),rn(oof_tabm),rn(oof_xgb)
print('\nIndividual OOF AUCs:')
print(f'  RealMLP: {roc_auc_score(y,oof_rmlp):.5f}')
print(f'  TabM:    {roc_auc_score(y,oof_tabm):.5f}')
print(f'  XGB:     {roc_auc_score(y,oof_xgb):.5f}')
print('\nCorrelations (rank-normalized OOF):')
print(f'  RealMLP-XGB:  {np.corrcoef(r_rmlp,r_xgb)[0,1]:.4f}')
print(f'  RealMLP-TabM: {np.corrcoef(r_rmlp,r_tabm)[0,1]:.4f}')
print(f'  XGB-TabM:     {np.corrcoef(r_xgb,r_tabm)[0,1]:.4f}')

oofs=[r_rmlp,r_xgb,r_tabm]; nm=['RealMLP','XGB','TabM']
def ba(w):
    w=np.array(w); w=w/w.sum(); return roc_auc_score(y,sum(wi*o for wi,o in zip(w,oofs)))
wts=np.ones(3)/3; best=ba(wts)
for _ in range(200):
    imp=False
    for i in range(3):
        for d in [0.05,-0.05,0.02,-0.02,0.01,-0.01]:
            tw=wts.copy(); tw[i]=max(0,tw[i]+d); a=ba(tw)
            if a>best+1e-7: best=a; wts=tw; imp=True
    if not imp: break
wts=wts/wts.sum()
print('\n3-way blend weights:')
for n,w in zip(nm,wts): print(f'  {n}: {w:.3f}')
print(f'\n3-way blend OOF: {best:.5f}')
print(f'(RealMLP alone: {rmlp_auc:.5f} | current best LB blend: 0.95390)')

# ---------- Submission ----------
pr=[rn(pred_rmlp),rn(pred_xgb),rn(pred_tabm)]
pb=sum(w*p for w,p in zip(wts,pr))
sub=pd.DataFrame({ID:test_id,TARGET:pb}); sub.to_csv('/kaggle/working/submission.csv',index=False)
print(f'\nSaved submission.csv | mean={pb.mean():.5f}')
print('Also saved oof_realmlp_ms5.npy, pred_realmlp_ms5.npy (RealMLP recovered)')

PyTorch: 2.10.0+cu128 | device: cuda
TabM and XGB prediction files confirmed present.
Loaded: (439140, 16) (188165, 15) (101371, 16)
After FE: (439140, 41) | cat: 20

=== RealMLP SEED 42 (1/3) ===
    epoch 1/5 score=0.92729 best=0.92729
    epoch 2/5 score=0.95244 best=0.95244
    epoch 3/5 score=0.95399 best=0.95399
    epoch 4/5 score=0.95485 best=0.95485
    epoch 5/5 score=0.95481 best=0.95485
  Seed 42 Fold 1: 0.95485
    epoch 1/5 score=0.92276 best=0.92276
    epoch 2/5 score=0.95005 best=0.95005
    epoch 3/5 score=0.95177 best=0.95177
    epoch 4/5 score=0.95250 best=0.95250
    epoch 5/5 score=0.95237 best=0.95250
  Seed 42 Fold 2: 0.95250
    epoch 1/5 score=0.92709 best=0.92709
    epoch 2/5 score=0.95146 best=0.95146
    epoch 3/5 score=0.95256 best=0.95256
    epoch 4/5 score=0.95337 best=0.95337
    epoch 5/5 score=0.95319 best=0.95337
  Seed 42 Fold 3: 0.95337
    epoch 1/5 score=0.92448 best=0.92448
    epoch 2/5 score=0.95020 best=0.95020
    epoch 3/5 score=0.95161 